# MAFIA Observer Training on Google Colab

**Features:**
- Auto-resume from latest checkpoint
- Saves to Google Drive (persistent)
- Uses T4 GPU (~2-3x faster than M1 Pro)

**Usage:**
1. Upload `mafia_colab_full.zip` to Google Drive root
2. Run all cells
3. If disconnected, just re-run - will auto-resume!

In [ ]:
# ============================================================
# CELL 1: Mount Drive & Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ============================================================
# CELL 2: Unzip & Setup Paths
# ============================================================
import os

# Path to uploaded zip file on Drive
ZIP_FILE = "/content/drive/MyDrive/mafia_colab_full.zip"

# Working directory (on Colab's fast storage)
WORK_DIR = "/content/MAFIA"

# Output directory (on Drive for persistence)
DRIVE_OUTPUT = "/content/drive/MyDrive/MAFIA_output"

# Unzip if not already done
if not os.path.exists(f"{WORK_DIR}/config.py"):
    print("Extracting MAFIA files...")
    !unzip -q "{ZIP_FILE}" -d {WORK_DIR}
    print("Done!")
else:
    print("MAFIA files already extracted.")

# Create Drive output directory
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Symlink output to Drive (so checkpoints persist)
LOCAL_OUTPUT = f"{WORK_DIR}/observer_offline"
if os.path.islink(LOCAL_OUTPUT):
    os.unlink(LOCAL_OUTPUT)
elif os.path.exists(LOCAL_OUTPUT):
    # Move existing checkpoints to Drive first
    !cp -rn {LOCAL_OUTPUT}/* {DRIVE_OUTPUT}/ 2>/dev/null || true
    !rm -rf {LOCAL_OUTPUT}

os.symlink(DRIVE_OUTPUT, LOCAL_OUTPUT)
print(f"Output symlinked: {LOCAL_OUTPUT} -> {DRIVE_OUTPUT}")

# Show structure
print("\nDirectory structure:")
!ls -la {WORK_DIR}

In [ ]:
# ============================================================
# CELL 3: Configure Python Path
# ============================================================
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

# Verify imports
try:
    from config import Config
    from RL_controller.mafia_observer import MAFIAObserver
    print("Imports OK!")
except ImportError as e:
    print(f"Import error: {e}")

In [ ]:
# ============================================================
# CELL 4: Check Resume State
# ============================================================
import torch as th

def check_resume_state():
    checkpoint_dir = f"{DRIVE_OUTPUT}/checkpoints"
    
    print("=" * 60)
    print("RESUME STATE")
    print("=" * 60)
    
    if not os.path.exists(checkpoint_dir):
        print("No checkpoints found. Will start fresh.")
        return
    
    # Completed iterations
    completed = [f for f in os.listdir(checkpoint_dir) 
                 if f.startswith("observer_best_") and f.endswith(".pth")]
    if completed:
        print(f"Completed: {sorted(completed)}")
    
    # In-progress
    for d in os.listdir(checkpoint_dir):
        if d.startswith("temp_iter_"):
            temp_dir = os.path.join(checkpoint_dir, d)
            latest = os.path.join(temp_dir, "latest_checkpoint.pth")
            
            if os.path.exists(latest):
                ckpt = th.load(latest, map_location="cpu")
                epoch = ckpt.get("epoch", -1)
                print(f"\nIn-progress: {d}")
                print(f"  Last epoch: {epoch}")
                print(f"  Will resume from epoch {epoch + 1}")
            else:
                # Check for best checkpoints
                best_ckpts = [f for f in os.listdir(temp_dir) 
                              if f.startswith("epoch_") and f.endswith(".pth")]
                if best_ckpts:
                    epochs = [int(f.split("_")[1].split(".")[0]) for f in best_ckpts]
                    max_epoch = max(epochs)
                    print(f"\nIn-progress: {d}")
                    print(f"  Best checkpoint: epoch {max_epoch}")
                    print(f"  Will resume from epoch {max_epoch + 1}")
    
    print("=" * 60)

check_resume_state()

In [ ]:
# ============================================================
# CELL 5: Training Configuration
# ============================================================

# === EDIT THESE IF NEEDED ===
START_YEAR = 2015
FIRST_INFER_YEAR = 2018
LAST_INFER_YEAR = 2022
EPOCHS = 50           # Epochs per iteration (base training)
BATCHES = None        # Auto-compute if None
# ===========================

DATA_DIR = f"{WORK_DIR}/data"
OUTPUT_DIR = f"{WORK_DIR}/observer_offline"  # Symlinked to Drive

print(f"Data: {DATA_DIR}")
print(f"Output: {OUTPUT_DIR} -> {DRIVE_OUTPUT}")
print(f"\nTraining: {START_YEAR} -> {LAST_INFER_YEAR}")
print(f"Epochs/iteration: {EPOCHS}")

In [ ]:
# ============================================================
# CELL 6: Verify Data
# ============================================================
import pandas as pd

# Find stock data
stock_files = ["stock_data_dynamic143.csv", "stock_data_top143.csv", 
               "stock_data_top23.csv", "stock_data.csv"]

stock_file = None
for f in stock_files:
    if os.path.exists(os.path.join(DATA_DIR, f)):
        stock_file = f
        break

if stock_file:
    df = pd.read_csv(os.path.join(DATA_DIR, stock_file))
    n_stocks = df['stock'].nunique()
    date_range = f"{df['date'].min()} to {df['date'].max()}"
    print(f"Stock data: {stock_file}")
    print(f"  Stocks: {n_stocks}")
    print(f"  Date range: {date_range}")
    print(f"  Rows: {len(df):,}")
else:
    print("ERROR: No stock data found!")

# Market data
market_file = "VNINDEX_1d_index.csv" if os.path.exists(f"{DATA_DIR}/VNINDEX_1d_index.csv") else "vnindex_data.csv"
if os.path.exists(f"{DATA_DIR}/{market_file}"):
    print(f"\nMarket data: {market_file}")
else:
    print("\nWARNING: No market data found")

In [ ]:
# ============================================================
# CELL 7: Patch Config for Colab
# ============================================================
from config import Config

_original_init = Config.__init__

def _patched_init(self, *args, **kwargs):
    kwargs["create_dirs"] = False
    _original_init(self, *args, **kwargs)
    self.dataDir = DATA_DIR
    self.stock_data_file = stock_file
    self.index_data_file = market_file

Config.__init__ = _patched_init
print("Config patched for Colab")

In [ ]:
# ============================================================
# CELL 8: START TRAINING (Auto-Resume)
# ============================================================
from scripts.train_observer_offline import run_offline_observer_training

print("=" * 70)
print("STARTING OBSERVER TRAINING")
print("=" * 70)
print(f"Output: {OUTPUT_DIR}")
print("Training will auto-resume from checkpoint if available")
print("=" * 70)

results = run_offline_observer_training(
    start_year=START_YEAR,
    first_infer_year=FIRST_INFER_YEAR,
    last_infer_year=LAST_INFER_YEAR,
    output_dir=OUTPUT_DIR,
    num_epochs=EPOCHS,
    batches_per_epoch=BATCHES,
    seed=2025,
    verbose=True,
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print(f"Checkpoints saved to: {DRIVE_OUTPUT}")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 9: View Results
# ============================================================
import json
import glob

# Training summary
summary_file = f"{OUTPUT_DIR}/offline_training_summary.json"
if os.path.exists(summary_file):
    with open(summary_file) as f:
        summary = json.load(f)
    
    print("Training Summary:")
    print("-" * 50)
    for r in summary:
        status = r.get('status', 'unknown')
        emoji = '' if 'success' in status else ''
        print(f"{emoji} Iteration {r.get('iteration', '?')}: {status}")

# Checkpoints
print("\nCheckpoints:")
!ls -lh {OUTPUT_DIR}/checkpoints/*.pth 2>/dev/null || echo "No final checkpoints yet"

# Validation metrics
print("\nValidation Metrics (last 3 epochs per iteration):")
for csv_path in sorted(glob.glob(f"{OUTPUT_DIR}/iter_*/valid_metrics.csv")):
    iter_name = os.path.basename(os.path.dirname(csv_path))
    df = pd.read_csv(csv_path)
    print(f"\n{iter_name}:")
    print(df[["epoch", "ces_score", "topk_sharpe_ratio", "direction_f1_macro"]].tail(3).to_string(index=False))

In [ ]:
# ============================================================
# CELL 10: TensorBoard (Optional)
# ============================================================
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tensorboard